**© Copyright AIDENTIFY. All rights reserved.**

# Part 2 | Session 12a: LoRA vs Full Fine-tuning 이론 비교

## 🎯 LoRA vs Full Fine-tuning 비교

이번 세션에서는 **LoRA**와 **Full Fine-tuning(FFT)**을 이론적, 실험적으로 비교합니다.

### 두 방법의 핵심 차이

| 구분 | LoRA | Full Fine-tuning |
|------|------|------------------|
| 학습 파라미터 | 전체의 ~1% | 전체 100% |
| 메모리 사용 | 낮음 (4bit + LoRA) | 높음 (FP16/FP32) |
| 학습 속도 | 빠름 | 느림 |
| 저장 크기 | ~수십 MB (어댑터만) | ~수 GB (전체 모델) |
| 성능 | 약간 낮을 수 있음 | 최고 성능 가능 |
| RTX 4060 가능 모델 | ~7B (QLoRA) | ~1.5B |

### 학습 목표

- ✅ LoRA와 FFT의 파라미터 수 차이를 직접 확인
- ✅ 메모리 사용량 비교 (GPU 모니터링)
- ✅ 학습 속도 비교
- ✅ 저장 크기 비교
- ✅ 상황별 선택 가이드 이해

## 1️⃣ 파라미터 수 비교 (전체 vs LoRA 학습 파라미터)

먼저 Qwen2.5-1.5B 모델의 전체 파라미터 수와 LoRA가 학습하는 파라미터 수를 비교합니다.

In [1]:
# 필수 라이브러리
import torch
import gc
import os
import sys
import time

# DeepSpeed가 단일 GPU FFT/LoRA에서도 accelerate에 의해 import되는데,
# CUDA 툴킷(nvcc/CUDA_HOME)이 없으면 op 컴파일 검사에서 MissingCUDAException이 발생한다.
# 단일 GPU에서는 DeepSpeed 커널이 필요 없으므로 CUDA 감지를 건너뛴다.
# (transformers/deepspeed import 전에 설정해야 적용됨)
os.environ.setdefault("DS_IGNORE_CUDA_DETECTION", "1")

# CUDA 메모리 단편화 완화 (8GB GPU에서 FFT 학습 시 OOM 여유 확보).
# CUDA 초기화 이전(첫 텐서 할당 전)에 설정해야 적용되므로 여기서 지정한다.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")

/home/hpe/LLM_master_5parts/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 라이브러리 임포트 완료


In [2]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

def get_gpu_memory_gb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**3
    return 0

print_gpu_memory("시작")

[시작] GPU: 0.0GB / 7.6GB


In [3]:
# Qwen2.5-1.5B 파라미터 분석
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

print("📊 Qwen2.5-1.5B 모델 파라미터 분석")
print("="*60)

# 1.5B 모델 구조 정보 (문서 기반)
model_info = {
    "총 파라미터": 1_500_000_000,
    "hidden_size": 1536,
    "num_layers": 28,
    "num_attention_heads": 12,
    "intermediate_size": 8960,
}

# LoRA 파라미터 계산
r = 16
hidden_size = model_info["hidden_size"]
num_layers = model_info["num_layers"]

# 어텐션 레이어: q, k, v, o_proj (4개)
attn_lora_params = 2 * r * hidden_size * 4 * num_layers

# FFN 레이어: gate, up, down_proj (3개) - intermediate_size 사용
intermediate_size = model_info["intermediate_size"]
ffn_lora_gate_up = 2 * r * intermediate_size * 2 * num_layers  # gate, up
ffn_lora_down = 2 * r * hidden_size * 1 * num_layers  # down

# 전체 LoRA 파라미터
attn_only_lora = attn_lora_params
full_lora = attn_lora_params + ffn_lora_gate_up + ffn_lora_down

total_params = model_info["총 파라미터"]

print(f"\n🔹 전체 모델 파라미터: {total_params:,} ({total_params/1e9:.1f}B)")
print(f"\n🔹 LoRA (어텐션만, r={r}):")
print(f"   학습 파라미터: {attn_only_lora:,} ({attn_only_lora/total_params*100:.3f}%)")
print(f"\n🔹 LoRA (어텐션+FFN, r={r}):")
print(f"   학습 파라미터: {full_lora:,} ({full_lora/total_params*100:.3f}%)")
print(f"\n🔹 Full Fine-tuning:")
print(f"   학습 파라미터: {total_params:,} (100.000%)")
print(f"\n📌 LoRA는 전체의 약 {full_lora/total_params*100:.2f}%만 학습! ({total_params//full_lora}배 적음)")

📊 Qwen2.5-1.5B 모델 파라미터 분석

🔹 전체 모델 파라미터: 1,500,000,000 (1.5B)

🔹 LoRA (어텐션만, r=16):
   학습 파라미터: 5,505,024 (0.367%)

🔹 LoRA (어텐션+FFN, r=16):
   학습 파라미터: 22,937,600 (1.529%)

🔹 Full Fine-tuning:
   학습 파라미터: 1,500,000,000 (100.000%)

📌 LoRA는 전체의 약 1.53%만 학습! (65배 적음)


## 2️⃣ LoRA 실습: 모델 로드 → LoRA 적용 → 파라미터 확인

In [4]:
# 공통 데이터 준비 (간단한 텍스트 데이터)
sample_texts = [
    "인공지능은 인간의 지능을 모방하여 학습, 추론, 판단 등의 작업을 수행하는 시스템입니다.",
    "머신러닝은 데이터로부터 패턴을 학습하는 알고리즘을 연구하는 분야입니다.",
    "딥러닝은 인공 신경망을 여러 층으로 쌓아 복잡한 패턴을 학습합니다.",
    "트랜스포머는 셀프 어텐션 메커니즘을 핵심으로 하는 아키텍처입니다.",
    "LoRA는 적은 파라미터만 학습하면서도 좋은 성능을 달성할 수 있습니다.",
] * 10  # 50개로 복제

print(f"✅ 샘플 데이터 준비 완료: {len(sample_texts)}개")

✅ 샘플 데이터 준비 완료: 50개


In [5]:
# LoRA 모델 로드 (4bit 양자화)
print("="*60)
print("📊 LoRA 모델 로드")
print("="*60)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch.cuda.empty_cache()
gc.collect()
mem_before_lora = get_gpu_memory_gb()

lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
lora_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
lora_model.enable_input_require_grads()

mem_after_load = get_gpu_memory_gb()
print(f"📌 4bit 모델 로드 GPU 메모리: {mem_after_load:.1f}GB")

📊 LoRA 모델 로드
📌 4bit 모델 로드 GPU 메모리: 0.4GB


In [6]:
# LoRA 어댑터 적용
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

mem_after_lora = get_gpu_memory_gb()
print(f"\n📌 LoRA 적용 후 GPU 메모리: {mem_after_lora:.1f}GB")
print(f"📌 LoRA 추가 메모리: {mem_after_lora - mem_after_load:.2f}GB")

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497

📌 LoRA 적용 후 GPU 메모리: 0.5GB
📌 LoRA 추가 메모리: 0.03GB


In [7]:
# LoRA 모델의 레이어 구조 확인
print("📊 LoRA 모델 레이어 구조 (첫 번째 Transformer 블록)")
print("="*60)
for name, param in lora_model.named_parameters():
    if "layers.0." in name:
        status = "🟢 학습" if param.requires_grad else "🔴 동결"
        print(f"  {status} {name}: {param.shape}, {param.dtype}")

📊 LoRA 모델 레이어 구조 (첫 번째 Transformer 블록)
  🔴 동결 base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight: torch.Size([401408, 1]), torch.uint8
  🔴 동결 base_model.model.model.layers.0.self_attn.q_proj.base_layer.bias: torch.Size([896]), torch.float16
  🟢 학습 base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: torch.Size([16, 896]), torch.float32
  🟢 학습 base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: torch.Size([896, 16]), torch.float32
  🔴 동결 base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight: torch.Size([57344, 1]), torch.uint8
  🔴 동결 base_model.model.model.layers.0.self_attn.k_proj.base_layer.bias: torch.Size([128]), torch.float16
  🟢 학습 base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: torch.Size([16, 896]), torch.float32
  🟢 학습 base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight: torch.Size([128, 16]), torch.float32
  🔴 동결 base_model.model.model.layers.0.self_attn.v_proj.base_l

## 3️⃣ FFT 실습: 모델 로드 → 전체 파라미터 학습 설정

⚠️ RTX 4060에서 FFT는 1.5B까지만 가능합니다. FP16으로 로드합니다.

In [8]:
# LoRA 모델 메모리 해제
lora_mem_usage = get_gpu_memory_gb()
del lora_model
gc.collect()
torch.cuda.empty_cache()
print("✅ LoRA 모델 메모리 해제")
print_gpu_memory("LoRA 해제 후")

✅ LoRA 모델 메모리 해제
[LoRA 해제 후] GPU: 0.0GB / 7.6GB


In [9]:
# FFT 모델 로드 (FP16 - 양자화 없음)
print("="*60)
print("📊 FFT 모델 로드 (FP16)")
print("="*60)

mem_before_fft = get_gpu_memory_gb()

fft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

mem_after_fft = get_gpu_memory_gb()
print(f"📌 FP16 모델 로드 GPU 메모리: {mem_after_fft:.1f}GB")

📊 FFT 모델 로드 (FP16)


`torch_dtype` is deprecated! Use `dtype` instead!


📌 FP16 모델 로드 GPU 메모리: 0.9GB


In [10]:
# FFT: 전체 파라미터 학습 가능으로 설정
trainable_params = 0
all_params = 0
for name, param in fft_model.named_parameters():
    all_params += param.numel()
    param.requires_grad = True  # 전체 파라미터 학습
    trainable_params += param.numel()

print(f"📊 FFT 파라미터 정보")
print(f"📌 전체 파라미터: {all_params:,}")
print(f"📌 학습 파라미터: {trainable_params:,}")
print(f"📌 학습 비율: {trainable_params/all_params*100:.1f}%")
print_gpu_memory("FFT 설정 후")

📊 FFT 파라미터 정보
📌 전체 파라미터: 494,032,768
📌 학습 파라미터: 494,032,768
📌 학습 비율: 100.0%
[FFT 설정 후] GPU: 0.9GB / 7.6GB


In [11]:
# FFT 모델 레이어 구조 확인
print("📊 FFT 모델 레이어 구조 (첫 번째 Transformer 블록)")
print("="*60)
for name, param in fft_model.named_parameters():
    if "layers.0." in name:
        status = "🟢 학습" if param.requires_grad else "🔴 동결"
        print(f"  {status} {name}: {param.shape}, {param.dtype}")

📊 FFT 모델 레이어 구조 (첫 번째 Transformer 블록)
  🟢 학습 model.layers.0.self_attn.q_proj.weight: torch.Size([896, 896]), torch.float16
  🟢 학습 model.layers.0.self_attn.q_proj.bias: torch.Size([896]), torch.float16
  🟢 학습 model.layers.0.self_attn.k_proj.weight: torch.Size([128, 896]), torch.float16
  🟢 학습 model.layers.0.self_attn.k_proj.bias: torch.Size([128]), torch.float16
  🟢 학습 model.layers.0.self_attn.v_proj.weight: torch.Size([128, 896]), torch.float16
  🟢 학습 model.layers.0.self_attn.v_proj.bias: torch.Size([128]), torch.float16
  🟢 학습 model.layers.0.self_attn.o_proj.weight: torch.Size([896, 896]), torch.float16
  🟢 학습 model.layers.0.mlp.gate_proj.weight: torch.Size([4864, 896]), torch.float16
  🟢 학습 model.layers.0.mlp.up_proj.weight: torch.Size([4864, 896]), torch.float16
  🟢 학습 model.layers.0.mlp.down_proj.weight: torch.Size([896, 4864]), torch.float16
  🟢 학습 model.layers.0.input_layernorm.weight: torch.Size([896]), torch.float16
  🟢 학습 model.layers.0.post_attention_layernorm.weight: torch.S

## 4️⃣ 메모리 사용량 비교 (GPU 모니터링)

In [12]:
# FFT 메모리 해제
fft_mem_usage = get_gpu_memory_gb()
del fft_model
gc.collect()
torch.cuda.empty_cache()

# 메모리 비교 결과
print("="*60)
print("📊 GPU 메모리 사용량 비교")
print("="*60)
print(f"\n🔹 LoRA (4bit + LoRA):  ~{lora_mem_usage:.1f}GB")
print(f"🔹 FFT (FP16):          ~{fft_mem_usage:.1f}GB")
print(f"\n📌 FFT는 LoRA 대비 약 {fft_mem_usage/max(lora_mem_usage, 0.1):.1f}배 메모리 사용")
print(f"📌 RTX 4060 (8GB)에서:")
print(f"   - LoRA: ✅ 여유 있음 (학습 시 ~{lora_mem_usage+1:.1f}GB 예상)")
print(f"   - FFT:  ⚠️ 빠듯함 (학습 시 ~{fft_mem_usage+2:.1f}GB 예상)")

📊 GPU 메모리 사용량 비교

🔹 LoRA (4bit + LoRA):  ~0.5GB
🔹 FFT (FP16):          ~0.9GB

📌 FFT는 LoRA 대비 약 2.0배 메모리 사용
📌 RTX 4060 (8GB)에서:
   - LoRA: ✅ 여유 있음 (학습 시 ~1.5GB 예상)
   - FFT:  ⚠️ 빠듯함 (학습 시 ~2.9GB 예상)


## 5️⃣ 학습 속도 비교 (동일 데이터, 동일 에포크)

동일한 데이터와 설정으로 LoRA와 FFT의 학습 속도를 비교합니다.

In [13]:
# 공통 데이터셋 준비
def prepare_dataset(tokenizer, texts, max_length=512):
    """텍스트를 토큰화하여 데이터셋 생성"""
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return Dataset.from_dict(tokenized)

dataset = prepare_dataset(tokenizer, sample_texts, max_length=256)
print(f"✅ 데이터셋 준비 완료: {len(dataset)}개 샘플")

✅ 데이터셋 준비 완료: 50개 샘플


In [14]:
# LoRA 학습 시간 측정
print("="*60)
print("⏱️ LoRA 학습 시간 측정")
print("="*60)

# LoRA 모델 재로드
lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
lora_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
lora_model.enable_input_require_grads()
lora_model = get_peft_model(lora_model, lora_config)

lora_training_args = TrainingArguments(
    output_dir="./output/lora_speed_test",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    bf16=True,
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

print("🚀 LoRA 학습 시작...")
lora_start = time.time()
lora_result = lora_trainer.train()
lora_time = time.time() - lora_start
lora_loss = lora_result.training_loss

print(f"\n✅ LoRA 학습 완료")
print(f"📌 소요 시간: {lora_time:.1f}초")
print(f"📌 Training Loss: {lora_loss:.4f}")
print_gpu_memory("LoRA 학습 후")

# 메모리 해제
lora_peak_mem = get_gpu_memory_gb()
del lora_model, lora_trainer
gc.collect()
torch.cuda.empty_cache()

⏱️ LoRA 학습 시간 측정


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


🚀 LoRA 학습 시작...


Step,Training Loss
5,2.409600



✅ LoRA 학습 완료
📌 소요 시간: 14.2초
📌 Training Loss: 2.3305
[LoRA 학습 후] GPU: 0.5GB / 7.6GB


In [19]:
# FFT 학습 시간 측정
# ⚠️ FFT(전체 파라미터 학습)는 1.5B + 기본 AdamW 기준 ~24GB가 필요하다
#    (가중치 3GB + 그래디언트 3GB + 옵티마이저 상태 ~18GB).
#    8GB GPU에 맞추기 위해 옵티마이저를 paged_adamw_8bit로 바꾼다:
#      • 8비트 AdamW  → 옵티마이저 상태 18GB → ~3GB
#      • paged       → 메모리 스파이크 시 옵티마이저 상태를 CPU RAM으로 자동 페이징
#    덕분에 9GB짜리 작업도 8GB GPU에서 OOM 없이 돌아간다(대신 CPU 오프로드로 다소 느림).
#    그래도 들어가지 않으면 OOM을 잡아 결과로 정리한다
#    (= "FFT는 8GB에 안 들어간다"가 이 실습의 결론).
print("="*60)
print("⏱️ FFT 학습 시간 측정")
print("="*60)

fft_oom = False
fft_time = fft_loss = fft_peak_mem = None

# FFT 모델 로드 (bf16 - 학습 인자의 bf16=True와 dtype 일치)
fft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
fft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
fft_model.config.use_cache = False

fft_training_args = TrainingArguments(
    output_dir="./output/fft_speed_test",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    bf16=True,
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)


fft_trainer = Trainer(
    model=fft_model,
    args=fft_training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

print("🚀 FFT 학습 시작...")
try:
    fft_start = time.time()
    fft_result = fft_trainer.train()
    fft_time = time.time() - fft_start
    fft_loss = fft_result.training_loss
    fft_peak_mem = get_gpu_memory_gb()

    print(f"\n✅ FFT 학습 완료")
    print(f"📌 소요 시간: {fft_time:.1f}초")
    print(f"📌 Training Loss: {fft_loss:.4f}")
    print_gpu_memory("FFT 학습 후")
except torch.cuda.OutOfMemoryError:
    fft_oom = True
    fft_peak_mem = 24.0  # 1.5B + 기본 AdamW 기준 이론적 추정치(참고용)
    print("\n💥 CUDA Out of Memory! — FFT 학습이 8GB GPU에 들어가지 않았습니다.")
    print("   👉 이것이 바로 LoRA가 필요한 이유입니다:")
    print("      • FFT(1.5B): 가중치 3GB + 그래디언트 3GB + 옵티마이저 상태 → 8GB 초과")
    print("      • LoRA(4bit): 위에서 보았듯 8GB 안에서 여유롭게 학습 완료")
    # 메모리 정리 (다음 셀에서 fft_model 참조 전에 안전하게 해제)
    del fft_trainer
    try:
        del fft_model
    except NameError:
        pass
    gc.collect()
    torch.cuda.empty_cache()

⏱️ FFT 학습 시간 측정


The model is already on multiple devices. Skipping the move to device specified in `args`.


🚀 FFT 학습 시작...


Step,Training Loss
5,1.022800



✅ FFT 학습 완료
📌 소요 시간: 7.9초
📌 Training Loss: 0.7637
[FFT 학습 후] GPU: 3.0GB / 7.6GB


## 6️⃣ 저장 크기 비교

In [16]:
# 모델 저장 및 크기 비교
import shutil

# FFT 모델 저장 (학습이 성공한 경우에만; OOM이면 이론값으로 대체)
fft_save_path = "./output/fft_speed_test/saved_model"
if not fft_oom:
    print("⏳ FFT 모델 저장 중...")
    fft_model.save_pretrained(fft_save_path)

    # FFT 저장 크기 계산
    fft_size = 0
    for root, dirs, files in os.walk(fft_save_path):
        for f in files:
            fft_size += os.path.getsize(os.path.join(root, f))

    # 메모리 해제
    del fft_model, fft_trainer
    gc.collect()
    torch.cuda.empty_cache()
else:
    # FFT 학습이 OOM으로 생략됨 → 1.5B(bf16, 2바이트/파라미터) 기준 이론적 저장 크기
    print("⚠️ FFT 학습이 OOM으로 생략됨 → 저장 크기는 이론값(1.5B × 2B ≈ 3GB)으로 표기")
    fft_size = int(1.5e9 * 2)

# LoRA 모델 저장 (재로드 필요)
print("⏳ LoRA 어댑터 저장을 위해 모델 재로드...")
# FFT 실험 후 VRAM 단편화로 device_map="auto"가 일부 레이어를 CPU로
# 오프로드하려다 4bit 양자화 제약에 걸린다(ValueError). 재로드 전 캐시를
# 비우고 device_map={"": 0}으로 전체를 GPU0에 고정해 CPU 오프로드를 차단한다.
gc.collect()
torch.cuda.empty_cache()
lora_model_2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)
lora_model_2 = get_peft_model(lora_model_2, lora_config)

lora_save_path = "./output/lora_speed_test/lora_adapter"
lora_model_2.save_pretrained(lora_save_path)

# LoRA 저장 크기 계산
lora_size = 0
for root, dirs, files in os.walk(lora_save_path):
    for f in files:
        lora_size += os.path.getsize(os.path.join(root, f))

# 크기 비교
print(f"\n📊 저장 크기 비교")
print(f"="*60)
print(f"🔹 LoRA 어댑터:  {lora_size/1024/1024:.1f} MB")
print(f"🔹 FFT 전체 모델: {fft_size/1024/1024:.1f} MB ({fft_size/1024/1024/1024:.2f} GB)")
print(f"\n📌 FFT는 LoRA 대비 약 {fft_size/max(lora_size, 1):.0f}배 큰 저장 공간 필요")

# 정리
del lora_model_2
gc.collect()
torch.cuda.empty_cache()

"""
# 임시 파일 정리
for path in ["./output/lora_speed_test", "./output/fft_speed_test"]:
    if os.path.exists(path):
        shutil.rmtree(path)
print("✅ 임시 파일 정리 완료")
"""

⏳ FFT 모델 저장 중...
⏳ LoRA 어댑터 저장을 위해 모델 재로드...

📊 저장 크기 비교
🔹 LoRA 어댑터:  33.6 MB
🔹 FFT 전체 모델: 942.3 MB (0.92 GB)

📌 FFT는 LoRA 대비 약 28배 큰 저장 공간 필요


'\n# 임시 파일 정리\nfor path in ["./output/lora_speed_test", "./output/fft_speed_test"]:\n    if os.path.exists(path):\n        shutil.rmtree(path)\nprint("✅ 임시 파일 정리 완료")\n'

## 7️⃣ 결과 분석 및 선택 가이드

In [17]:
# 종합 비교표
# FFT가 OOM으로 학습되지 않은 경우에도 표가 깨지지 않도록 안전하게 포맷
fft_mem_str  = f"~{fft_peak_mem:.1f}GB" if fft_peak_mem is not None else "N/A"
fft_time_str = f"{fft_time:.1f}초"      if fft_time is not None     else "OOM (8GB 초과)"
fft_loss_str = f"{fft_loss:.4f}"        if fft_loss is not None     else "N/A"

print("="*70)
print("📊 LoRA vs FFT 종합 비교 결과")
print("="*70)
print(f"\n{'항목':<20} {'LoRA':<25} {'FFT':<25}")
print("-"*70)
print(f"{'학습 파라미터':<20} {'~1% (수십만 개)':<25} {'100% (15억 개)':<25}")
print(f"{'GPU 메모리':<20} {f'~{lora_peak_mem:.1f}GB':<25} {fft_mem_str:<25}")
print(f"{'학습 시간 (1 epoch)':<20} {f'{lora_time:.1f}초':<25} {fft_time_str:<25}")
print(f"{'Training Loss':<20} {f'{lora_loss:.4f}':<25} {fft_loss_str:<25}")
print(f"{'저장 크기':<20} {f'{lora_size/1024/1024:.1f}MB':<25} {f'{fft_size/1024/1024:.0f}MB':<25}")
print(f"{'RTX 4060 최대 모델':<20} {'7B (QLoRA)':<25} {'1.5B':<25}")
print("-"*70)

if fft_oom:
    print("\n💡 FFT는 8GB GPU에서 OOM으로 학습되지 않았습니다.")
    print("   → 동일 모델을 LoRA(4bit)로는 학습할 수 있다는 점이 핵심 결론입니다.")

📊 LoRA vs FFT 종합 비교 결과

항목                   LoRA                      FFT                      
----------------------------------------------------------------------
학습 파라미터              ~1% (수십만 개)               100% (15억 개)             
GPU 메모리              ~0.5GB                    ~0.9GB                   
학습 시간 (1 epoch)      14.2초                     7.7초                     
Training Loss        2.3305                    0.7633                   
저장 크기                33.6MB                    942MB                    
RTX 4060 최대 모델       7B (QLoRA)                1.5B                     
----------------------------------------------------------------------


In [18]:
# 선택 가이드
print("\n📋 LoRA vs FFT 선택 가이드")
print("="*60)

print("""
🔹 LoRA를 선택해야 하는 경우:
   ✅ GPU 메모리가 제한적 (8GB 이하)
   ✅ 큰 모델을 사용해야 하는 경우 (7B+)
   ✅ 여러 태스크용 어댑터를 별도 관리하고 싶은 경우
   ✅ 빠른 실험/프로토타이핑이 필요한 경우
   ✅ 원본 모델을 보존하면서 커스터마이징하고 싶은 경우

🔹 FFT를 선택해야 하는 경우:
   ✅ 충분한 GPU 메모리가 있는 경우 (24GB+)
   ✅ 최고 성능이 필요한 경우
   ✅ 모델 구조를 크게 변경해야 하는 경우
   ✅ 데이터가 충분히 많은 경우
   ✅ 특정 도메인에 완전히 특화시켜야 하는 경우

📌 실무 권장사항:
   → 먼저 LoRA로 빠르게 실험하고,
   → 성능이 부족하면 FFT를 고려하세요!
""")


📋 LoRA vs FFT 선택 가이드

🔹 LoRA를 선택해야 하는 경우:
   ✅ GPU 메모리가 제한적 (8GB 이하)
   ✅ 큰 모델을 사용해야 하는 경우 (7B+)
   ✅ 여러 태스크용 어댑터를 별도 관리하고 싶은 경우
   ✅ 빠른 실험/프로토타이핑이 필요한 경우
   ✅ 원본 모델을 보존하면서 커스터마이징하고 싶은 경우

🔹 FFT를 선택해야 하는 경우:
   ✅ 충분한 GPU 메모리가 있는 경우 (24GB+)
   ✅ 최고 성능이 필요한 경우
   ✅ 모델 구조를 크게 변경해야 하는 경우
   ✅ 데이터가 충분히 많은 경우
   ✅ 특정 도메인에 완전히 특화시켜야 하는 경우

📌 실무 권장사항:
   → 먼저 LoRA로 빠르게 실험하고,
   → 성능이 부족하면 FFT를 고려하세요!



## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | LoRA | Full Fine-tuning |
|------|------|------------------|
| 학습 파라미터 | ~1% (수십만 개) | 100% (15억 개) |
| GPU 메모리 | 적음 (4bit + LoRA) | 많음 (FP16 전체) |
| 학습 속도 | 빠름 | 느림 |
| 저장 크기 | ~수십 MB (어댑터만) | ~수 GB (전체 모델) |

### 핵심 포인트

- 🎯 **LoRA는 전체 파라미터의 ~1%만 학습**하므로 메모리와 시간이 절약됩니다
- 🎯 **FFT는 최고 성능**을 달성할 수 있지만, 자원이 많이 필요합니다
- 🎯 **RTX 4060(8GB) 단일 GPU**에서 LoRA는 7B(QLoRA)까지, FFT는 1.5B까지 가능합니다
- 🎯 **실무 권장 순서**: 먼저 LoRA로 빠르게 실험 → 성능이 부족하면 FFT 고려

### 다음 단계

- ➡️ **Session 12b**: LoRA vs FFT 실전 비교 실습 - 동일 데이터로 학습 후 성능 비교
